# Perbandingan model dan evaluasi akhir AHI

Notebook ini menggabungkan sembilan kondisi CNN tersimpan, melatih baseline median dan Ridge, memilih berdasarkan validation, membekukan protokol, lalu mengevaluasi test. Tidak ada training ulang CNN. Seluruh 411 ID disertakan sesuai arahan klien; data dan target asli dipertahankan.

Tahapan: kontrol eksekusi; validasi artifacts dan baseline; pembekuan pemilihan; hasil test; kontribusi transfer learning; biaya dan grafik; pembahasan dan kesimpulan. Output lengkap tetap lokal di artifacts.

In [ ]:
from pathlib import Path
import os
import sys
root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src' / 'config.py').is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ['MPLBACKEND'] = 'Agg'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import json
import pandas as pd
from IPython.display import Image, Markdown, display
from src.comparison import run_comparison
from src.config import ARTIFACTS_DIR
output_root = ARTIFACTS_DIR / 'comparison'
RUN_FINAL_EVALUATION = False

## 1. Protokol dan baseline

Eksekusi awal PHASE 5 telah diizinkan pengguna. Flag default False agar membuka laporan hanya membaca hasil tersimpan. Runner menolak direktori hasil yang sudah ada; jangan menghapusnya untuk tuning ulang memakai test yang sama.

Hash checkpoint, konfigurasi, prediksi validation, history, dan urutan sampel sembilan CNN diperiksa. Baseline memakai flattened scaled_features float32, mask IMF, dan one-hot posisi dari sampel yang sama, tanpa ID atau target sebagai fitur. Jumlah fitur ditentukan arrays. Representasi baseline mempertahankan fitur sebelum kuantisasi citra, sehingga baseline menguji manfaat pipeline citra, bukan hanya arsitektur.

DummyRegressor memakai median target train. Ridge memakai StandardScaler yang fit hanya pada train; alpha 0.1, 1, 10, 100, 1000 dipilih dari RMSE validation, lalu MAE. Tidak ada refit train+validation. Semua pilihan CNN/baseline dan hash aset disimpan sebelum test NPZ pertama kali dimuat. CNN utama dipilih dari sembilan kondisi; pembanding tabular dilaporkan terpisah dan bersama.

In [ ]:
if RUN_FINAL_EVALUATION:
    run_comparison(output_root, test_evaluation_approved=True)
else:
    if not (output_root / 'completion.json').is_file():
        raise FileNotFoundError('Hasil PHASE 5 belum tersedia. Eksekusi awal memerlukan izin dan flag eksplisit.')
    print('Membaca hasil tersimpan; tidak melatih baseline/CNN atau mengulang evaluasi test.')
completion = json.loads((output_root / 'completion.json').read_text(encoding='utf-8'))
selection = json.loads((output_root / 'selection_validation.json').read_text(encoding='utf-8'))
assert completion['cnn_retrained'] is False and selection['selected_before_test'] is True
print('Status:', completion['status'], '| kondisi:', completion['frozen_conditions'])

## 2. Pemilihan berdasarkan validation

Kriteria tetap adalah RMSE validation terkecil, MAE jika seri, lalu label stabil. Tabel mencakup sembilan CNN dan dua baseline. Pemilihan tidak diubah setelah test. Pencarian alpha Ridge dan biaya training disimpan agar dapat diaudit.

In [ ]:
validation = pd.read_csv(output_root / 'comparison_validation.csv')
costs = pd.read_csv(output_root / 'training_costs.csv')
display(validation[['Model', 'Experiment', 'samples', 'MAE', 'MSE', 'RMSE', 'R2', 'negative_predictions']].round(4))
display(pd.read_csv(output_root / 'ridge_validation_search.csv').round(4))
for group, chosen in selection['selected'].items():
    print(group, ':', chosen['Model'], '/', chosen['Experiment'], '| validation RMSE:', round(chosen['RMSE'], 4))

## 3. Test sesudah pembekuan protokol

Tabel test melaporkan seluruh kondisi beku dalam urutan yang sama; tabel ini tidak menentukan pemenang atau tuning. Target ekstrem dan prediksi negatif dipertahankan. Metrik keseluruhan dan per posisi mencantumkan jumlah sampel.

Kontribusi transfer learning dihitung sebagai error sebelum dikurangi error sesudah; nilai positif berarti membaik. Persentase tidak didefinisikan jika error awal nol. Confidence interval dan signifikansi tidak dihitung; penelitian ini masih satu seed dan satu split.

In [ ]:
test = pd.read_csv(output_root / 'comparison_test.csv')
position_metrics = pd.read_csv(output_root / 'metrics_test.csv')
contributions = pd.read_csv(output_root / 'transfer_contributions.csv')
display(test[['Model', 'Experiment', 'samples', 'MAE', 'MSE', 'RMSE', 'R2', 'negative_predictions']].round(4))
display(position_metrics[position_metrics.position != 'Overall'][['Model', 'Experiment', 'position', 'samples', 'MAE', 'RMSE', 'R2', 'negative_predictions']].round(4))
display(contributions[contributions.position == 'Overall'].round(4))
print('Test:', completion['test_samples'], 'sampel,', completion['test_patients'], 'ID,', completion['test_components'], 'komponen; posisi:', completion['test_positions'])
chosen = selection['selected']['cnn']
chosen_test = test[(test.Model == chosen['Model']) & (test.Experiment == chosen['Experiment'])].iloc[0]
display(Markdown(f"CNN yang dipilih sebelum test: **{chosen['Model']}/{chosen['Experiment']}**, test MAE {chosen_test.MAE:.4f}, RMSE {chosen_test.RMSE:.4f}, R2 {chosen_test.R2:.4f}."))

## 4. Biaya, history, dan visualisasi

Grafik 300 DPI mencakup MAE/RMSE/R2, posisi, heatmap, scatter, biaya dan jumlah parameter, serta kurva training/validation tersimpan. Waktu kumulatif fine-tuning memasukkan feature extraction. Baseline memakai CPU dan pencarian Ridge; CNN memakai GPU. Batas epoch sama tidak berarti jumlah operasi atau parameter sama.

Scatter memakai sumbu bersama dan seluruh sampel, tanpa clipping kasus ekstrem. Warna heatmap dinormalisasi per metrik; angka tetap nilai asli. Grafik individual test, termasuk residual dan Bland-Altman deskriptif dengan catatan pengukuran berulang, tersimpan per kondisi.

Error Ridge validation mendominasi beberapa skala grafik semua kondisi. Panel pendamping khusus CNN disediakan agar perbedaan sembilan CNN tetap terbaca. Seluruh kondisi dan kasus tetap tersedia pada tabel dan grafik asli; panel ini hanya mengubah tampilan, bukan model, prediksi, atau pemilihan.

In [ ]:
display(costs[['Model', 'Experiment', 'compute_device', 'epochs', 'best_epoch', 'trainable_parameters', 'training_seconds', 'cumulative_training_seconds', 'peak_allocated_MiB']].round(3))
full_views = ['validation_metrics_heatmap.png', 'test_metrics_heatmap.png', 'validation_actual_vs_predicted_comparison.png', 'test_actual_vs_predicted_comparison.png']
for filename in full_views:
    display(Image(filename=str(output_root / 'figures' / filename), width=1000))
supplement = json.loads((output_root / 'supplemental_figures.json').read_text(encoding='utf-8'))
for filename in supplement['figures']:
    display(Image(filename=str(output_root / 'figures' / filename), width=1000))

## 5. Pembahasan berdasarkan hasil aktual

Pembahasan berikut dihasilkan dari artifacts aktual: pemilihan validation, kontribusi transfer learning, biaya, pola overfitting, posisi, dan keterbatasan data. Tidak ada klaim bahwa transfer learning selalu unggul atau bahwa evaluasi internal merupakan validasi klinis.

In [ ]:
display(Markdown((output_root / 'research_discussion.md').read_text(encoding='utf-8')))

## Kesimpulan

Gunakan pilihan yang telah dikunci dari validation untuk menyampaikan hasil test. Bandingkan CNN dengan baseline tabular serta laporkan hasil negatif dan per posisi. Test telah terpapar setelah eksekusi ini; perubahan berikutnya tidak boleh kembali memakai test sebagai validation sambil mengklaim evaluasi independen.

Seluruh 411 ID disertakan sesuai arahan klien; data dan target asli dipertahankan. Koreksi sumber memerlukan preprocessing dan eksperimen baru. Hasil ini belum menunjukkan generalisasi eksternal atau kesiapan klinis.